# AIC 2026 — Notebook 02: Retrieve · Fusion · Refine · Candidate Package (Local / 2×T4)

**Input**: artifact của NB01 (`manifest.json`, `records/*.parquet`, `index/*`).
**Output**: `review_package/` gồm `candidates.parquet`, `queries_parsed.json`, contact sheet ảnh, `candidates_manifest.json`
→ NB03 chỉ human-review + xuất submission, **không chạy lại model**.

Luồng: MiMo API parse/dịch/mở rộng query (text-only) → retrieval song ngữ đa nhánh (SigLIP2 visual / BGE-M3 dense caption·transcript·summary / BM25 OCR·caption·transcript / object soft-boost)
→ weighted RRF + video prior → temporal NMS + diversity → BGE-reranker → joint-score frame refinement → human review visual/Q&A → TRAKE alignment theo độ phủ event.

**GPU lifecycle**: mỗi stage GPU load 1 model → dùng → unload (`free_gpu()`); MiMo chạy qua OpenRouter API nên không chiếm VRAM. Qwen visual/answer được giữ như tùy chọn nhưng mặc định tắt.
**Checkpoint**: theo từng query (`ckpt/q_<qid>.json`) → resume sau timeout.

In [ ]:
# ============================================================
# CELL 1 — Env probe
# ============================================================
import sys, os, subprocess, importlib

def have(m):
    try:
        importlib.import_module(m); return True
    except Exception:
        return False

for mod, pkg in {"faiss": "faiss-cpu", "pyarrow": "pyarrow", "cv2": "opencv-python-headless"}.items():
    if not have(mod):
        print("pip install", pkg)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np, pandas as pd, faiss, torch, cv2, transformers
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| n_gpu", torch.cuda.device_count())
print("transformers", transformers.__version__, "| cv2", cv2.__version__)
# Query parser dùng MiMo qua OpenRouter; không cần tải Qwen hoặc kiểm tra Qwen3-VL.
for i in range(torch.cuda.device_count()):
    print(f"  GPU{i}: {torch.cuda.get_device_properties(i).name}")

In [ ]:
# ============================================================
# CELL 2 — CFG tập trung
# ============================================================
import os, glob, json, time, gc, re, math, pickle, unicodedata, hashlib, traceback
from pathlib import Path

CFG = dict(
    SEARCH_ROOTS = ["/kaggle/input", "."],
    ART_INPUT    = "/kaggle/input/notebooks/kitnehi1211/01-build-indices-local/artifacts",
    ART_OUT      = "/kaggle/working/artifacts02_mimo",
    DATASET_ROOT = None,                       # để refinement đọc video gốc
    QUERY_ROOT   = None,

    # --- model ---
    SIGLIP       = "google/siglip2-giant-opt-patch16-384",
    RERANK       = "BAAI/bge-reranker-v2-m3",
    BGE_M3       = "BAAI/bge-m3",
    QWEN         = "Qwen/Qwen3-VL-4B-Instruct",  # chỉ dự phòng; mặc định không load
    MIMO_MODEL   = "xiaomi/mimo-v2.5",
    LOCAL_MODEL_DIR = None,
    USE_QWEN_PARSE  = False,                   # đã thay bằng MiMo API
    USE_MIMO_PARSE  = True,                    # query planning / dịch / retrieval variants
    MIMO_DRY_RUN    = False,                   # True = không gọi API (full run phải False)
    MIMO_MAX_RETRIES = 3,
    MIMO_TIMEOUT     = 180,
    USE_QWEN_VERIFY = False,                   # human review thay visual verify
    USE_QWEN_ANSWER = False,                   # human review tự điền answer Q&A
    QWEN_4BIT       = False,                   # bật nếu FP16 không vừa VRAM
    GPU_MAIN     = 0,
    GPU_AUX      = 1,

    # --- retrieval top-k mỗi nhánh ---
    K_VISUAL     = 400,
    K_CAP_DENSE  = 300,
    K_TR_DENSE   = 200,
    K_SM_DENSE   = 60,
    K_BM25_OCR   = 300,
    K_BM25_CAP   = 200,
    K_BM25_TR    = 200,
    K_BM25_SM    = 60,

    # --- fusion (weighted RRF, k=60) ---
    RRF_K        = 60,
    W_DEFAULT    = dict(visual=1.00, cap_dense=0.70, tr_dense=0.55, bm25_ocr=0.45,
                        bm25_cap=0.35, bm25_tr=0.40, obj=0.15),
    W_OCR_HEAVY  = dict(visual=0.80, cap_dense=0.50, tr_dense=0.40, bm25_ocr=1.00,
                        bm25_cap=0.30, bm25_tr=0.35, obj=0.10),
    W_SPEECH     = dict(visual=0.65, cap_dense=0.45, tr_dense=1.00, bm25_ocr=0.25,
                        bm25_cap=0.30, bm25_tr=0.85, obj=0.10),
    W_VISUAL     = dict(visual=1.20, cap_dense=0.80, tr_dense=0.30, bm25_ocr=0.20,
                        bm25_cap=0.40, bm25_tr=0.20, obj=0.20),
    VIDEO_PRIOR_W   = 0.25,      # cộng thêm, không đè frame evidence
    VIDEO_PRIOR_CAP = 0.35,      # trần đóng góp của prior

    # --- diversity / NMS ---
    NMS_TIME_SEC    = 4.0,       # gộp candidate cùng video trong cửa sổ này
    MAX_PER_VIDEO_TOP20 = 3,     # top20 đầu: tối đa 3 frame / video
    MAX_PER_VIDEO_ALL   = 12,
    N_SUBMIT        = 100,

    # --- rerank / verify ---
    RERANK_TOPN     = 60,
    RERANK_W        = 0.6,       # trọng số blend vào fused score
    QWEN_VERIFY_TOPM = 12,
    QWEN_QA_TOPM     = 8,

    # --- frame refinement ---
    REFINE_TOPR     = 20,        # số candidate/query được refine tự động
    REFINE_WIN_SEC  = 3.0,
    REFINE_COARSE   = 5,         # bước frame ở vòng coarse
    REFINE_FINE_R   = 6,         # bán kính frame ở vòng fine
    REFINE_BATCH    = 16,

    # --- TRAKE ---
    TRAKE_VIDEOS    = 8,         # số video ứng viên đưa vào alignment
    TRAKE_PER_EVENT = 40,        # top frame / event / video
    TRAKE_BEAM      = 12,
    TRAKE_SEQ_PER_VIDEO = 6,
    TRAKE_MIN_GAP_SEC   = 0.15,

    SEED = 20260824,
    WALL_BUDGET_H = 10.0,
    LIMIT_QUERIES = None,        # int để smoke test
    REQUIRE_MIMO_PARSE = True,   # không âm thầm chạy rule parser cho full run
)

np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
T0 = time.time()

def budget_left(): return CFG["WALL_BUDGET_H"] * 3600 - (time.time() - T0)
def check_budget(tag=""):
    if budget_left() <= 0:
        raise TimeoutError(f"budget hết tại [{tag}] — Save Version rồi resume")

class Timer:
    def __init__(self, t): self.tag = t
    def __enter__(self): self.s = time.time(); print(f"[{self.tag}] start"); return self
    def __exit__(self, *a): print(f"[{self.tag}] {time.time()-self.s:.1f}s")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def jload(p, d=None):
    try:
        with open(p, encoding="utf-8") as f: return json.load(f)
    except Exception: return d

def jdump(o, p):
    p = str(p); os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p + ".tmp", "w", encoding="utf-8") as f:
        json.dump(o, f, ensure_ascii=False, indent=1)
    os.replace(p + ".tmp", p)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    except Exception:
        OPENROUTER_API_KEY = ""
if CFG["USE_MIMO_PARSE"] and not CFG["MIMO_DRY_RUN"] and not OPENROUTER_API_KEY:
    raise RuntimeError("Thiếu Kaggle Secret OPENROUTER_API_KEY cho MiMo parser")
print("CFG ok | budget %.2f h | MiMo API key: %s" % (budget_left() / 3600, bool(OPENROUTER_API_KEY)))

In [ ]:
# ============================================================
# CELL 3 — Load artifact NB01 (auto-discover) + text utils
# ============================================================
# Class này được pickle từ NB01 -> BẮT BUỘC định nghĩa lại (cùng tên) trước khi unpickle.
class BM25MultiField:
    """BM25 Okapi multi-field; định nghĩa phải khớp NB01 để pickle load được."""
    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.fields = {}
        self.n_doc = 0

    def add_field(self, field, docs_tokens, weight=1.0):
        postings, df, dl = {}, {}, np.zeros(len(docs_tokens), dtype=np.int32)
        for i, toks in enumerate(docs_tokens):
            dl[i] = len(toks)
            tf = {}
            for t in toks:
                tf[t] = tf.get(t, 0) + 1
            for t, c in tf.items():
                postings.setdefault(t, []).append((i, c))
                df[t] = df.get(t, 0) + 1
        self.fields[field] = dict(postings={t: np.array(v, dtype=np.int32) for t, v in postings.items()},
                                  df=df, dl=dl, avgdl=float(dl.mean()) if len(dl) else 1.0,
                                  N=len(docs_tokens), weight=float(weight))
        self.n_doc = max(self.n_doc, len(docs_tokens))
        return self

    def _idf(self, F, t):
        n = F["df"].get(t, 0)
        return math.log(1 + (F["N"] - n + 0.5) / (n + 0.5))

    def search(self, query_by_field, topk=200):
        scores = {}
        for field, toks in query_by_field.items():
            F = self.fields.get(field)
            if not F or not toks:
                continue
            w, k1, b = F["weight"], self.k1, self.b
            avgdl = max(F["avgdl"], 1e-6)
            for t in set(toks):
                pl = F["postings"].get(t)
                if pl is None:
                    continue
                idf = self._idf(F, t)
                docs, tf = pl[:, 0], pl[:, 1].astype(np.float32)
                dl = F["dl"][docs].astype(np.float32)
                s = w * idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
                for d, v in zip(docs, s):
                    scores[int(d)] = scores.get(int(d), 0.0) + float(v)
        if not scores:
            return np.zeros(0, np.int64), np.zeros(0, np.float32)
        items = sorted(scores.items(), key=lambda x: -x[1])[:topk]
        return (np.array([i for i, _ in items], np.int64),
                np.array([s for _, s in items], np.float32))

def bm25_load(d):
    """Dựng lại BM25 từ dict thuần do NB01 ghi (không phụ thuộc pickle class)."""
    bm = BM25MultiField(k1=d["k1"], b=d["b"])
    bm.fields = d["fields"]; bm.n_doc = d["n_doc"]
    return bm

def fast_glob(roots, pat, maxdepth=6, limit=None):
    """Quét theo từng tầng bằng glob KHÔNG recursive.
    glob(**, recursive=True) trên /kaggle/input đi bộ toàn bộ cây dataset (hàng chục phút);
    cách này chỉ liệt kê thư mục ở mỗi tầng nên nhanh hơn hàng trăm lần."""
    out = []
    for root in ([roots] if isinstance(roots, str) else roots):
        if not root or not os.path.isdir(root):
            continue
        for d in range(1, maxdepth + 1):
            out += glob.glob(os.path.join(root, *(["*"] * (d - 1)), pat))
            if limit and len(out) >= limit:
                return out[:limit]
    return out

def find_art():
    if CFG["ART_INPUT"]:
        return CFG["ART_INPUT"]
    cands = fast_glob(CFG["SEARCH_ROOTS"], "manifest.json", maxdepth=6)
    cands = [c for c in cands if (jload(c, {}) or {}).get("notebook") == "01_build_indices_local"]
    cands.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    if not cands:
        raise FileNotFoundError("Không tìm thấy artifact NB01 — set CFG['ART_INPUT']")
    return os.path.dirname(cands[0])

ART = Path(find_art())
OUT = Path(CFG["ART_OUT"]); (OUT / "ckpt").mkdir(parents=True, exist_ok=True)
(OUT / "review_package" / "sheets").mkdir(parents=True, exist_ok=True)
MANIFEST01 = jload(ART / "manifest.json", {})
print("ART :", ART)
print("OUT :", OUT)
print("NB01 counts:", MANIFEST01.get("counts"))
print("NB01 integrity_all_pass:", MANIFEST01.get("integrity_all_pass"))

with Timer("load records + indices"):
    KF  = pd.read_parquet(ART / "records" / "keyframes.parquet")
    CAP = pd.read_parquet(ART / "records" / "caption.parquet")
    OCR = pd.read_parquet(ART / "records" / "ocr.parquet")
    TR  = pd.read_parquet(ART / "records" / "transcript.parquet")
    SM  = pd.read_parquet(ART / "records" / "summary.parquet")
    OBJ = pd.read_parquet(ART / "records" / "objects.parquet")

    SIG_INDEX  = faiss.read_index(str(ART / "index" / "siglip.faiss"))
    SIG_ROWMAP = pd.read_parquet(ART / "index" / "siglip_rowmap.parquet")
    CAP_INDEX  = faiss.read_index(str(ART / "index" / "cap_dense.faiss"))
    CAP_MAP    = pd.read_parquet(ART / "index" / "cap_dense_rowmap.parquet")
    TR_INDEX   = faiss.read_index(str(ART / "index" / "tr_dense.faiss"))
    TR_MAP     = pd.read_parquet(ART / "index" / "tr_dense_rowmap.parquet")
    SM_INDEX   = faiss.read_index(str(ART / "index" / "sm_dense.faiss"))
    SM_MAP     = pd.read_parquet(ART / "index" / "sm_dense_rowmap.parquet")

    BM = {}
    for nm in ("ocr", "caption", "transcript", "summary"):
        with open(ART / "index" / f"bm25_{nm}.pkl", "rb") as f:
            d = pickle.load(f)
        BM[nm] = dict(bm25=bm25_load(d["bm25"]), meta=d["meta"])
    with open(ART / "index" / "objects_inverted.pkl", "rb") as f:
        OBJ_IDX = pickle.load(f)

# --- lookup nhanh ---
KF["key"] = KF.video_id + ":" + KF.keyframe_n.astype(str)
KF_IDX = KF.set_index("key")
FPS_BY_VID = {k: float(v) for k, v in KF.dropna(subset=["fps"]).groupby("video_id")["fps"].first().items()}
PTS_BY_KEY = dict(zip(KF.key, KF.pts_time))
FI_BY_KEY  = dict(zip(KF.key, KF.frame_idx))
SUMMARY_BY_VID = dict(zip(SM.video_id, SM.summary_en))
OCR_BY_KEY = {f"{v}:{n}": t for v, n, t in zip(OCR.video_id, OCR.keyframe_n, OCR.text)}
CAP_BY_KEY = {f"{v}:{n}": t for v, n, t in zip(CAP.video_id, CAP.keyframe_n, CAP.text)}

# range hàng SigLIP2 theo video (rowmap được build tuần tự theo video)
VID_ROWS = {}
_g = SIG_ROWMAP.reset_index().groupby("video_id")["index"]
for v, s in _g:
    a = s.values
    VID_ROWS[v] = (int(a.min()), int(a.max()) + 1, bool(len(a) == a.max() - a.min() + 1))
print("videos in siglip rowmap:", len(VID_ROWS),
      "| contiguous:", sum(1 for x in VID_ROWS.values() if x[2]), "/", len(VID_ROWS))

_VN = str.maketrans({"đ": "d", "Đ": "D"})
def nfc(s): return unicodedata.normalize("NFC", s or "")
def strip_dia(s):
    s = unicodedata.normalize("NFD", (s or "").translate(_VN))
    return "".join(c for c in s if unicodedata.category(c) != "Mn")
TOKEN_RE = re.compile(r"[0-9]+(?:[.,:/][0-9]+)*|[^\W_]+", re.UNICODE)
def tokenize(s): return TOKEN_RE.findall(nfc(s).lower())
def tokenize_nodia(s): return TOKEN_RE.findall(strip_dia(nfc(s)).lower())

# BM25MultiField được unpickle từ NB01 -> class phải tồn tại trong namespace
print("BM25 class:", type(BM["ocr"]["bm25"]).__name__, "| n_doc:", BM["ocr"]["bm25"].n_doc)

In [ ]:
# ============================================================
# CELL 4 — Nạp bộ query (query-*.txt) + parse rule-based (fallback không cần model)
# ============================================================
def find_query_root():
    if CFG["QUERY_ROOT"]:
        return CFG["QUERY_ROOT"]
    hits = fast_glob(CFG["SEARCH_ROOTS"], "query-*.txt", maxdepth=6, limit=1)
    return os.path.dirname(hits[0]) if hits else None

def find_dataset_root():
    if CFG["DATASET_ROOT"]:
        return CFG["DATASET_ROOT"]
    r = (MANIFEST01.get("roots") or {}).get("dataset")   # NB01 đã dò và ghi vào manifest
    if r and os.path.isdir(r):
        return r
    hits = fast_glob(CFG["SEARCH_ROOTS"], "Videos_*", maxdepth=6, limit=1)
    return os.path.dirname(hits[0]) if hits else None

with Timer("resolve query/dataset root"):
    QROOT = find_query_root(); DROOT = find_dataset_root()
print("QUERY_ROOT :", QROOT)
print("DATASET_ROOT:", DROOT)

VIDEO_FILE = {}
with Timer("index video files"):
    for p in fast_glob(DROOT, "*.mp4", maxdepth=4):
        VIDEO_FILE[Path(p).stem] = p
print("video files :", len(VIDEO_FILE))

QTYPES = ("kis", "qa", "trake")
EVENT_RE = re.compile(r"^\s*(?:E|Event)\s*(\d+)\s*[:.\-)]?\s*(.+)$", re.IGNORECASE)

def load_queries():
    qs = []
    for p in sorted(set(fast_glob(QROOT or ".", "query-*.txt", maxdepth=3))):
        stem = Path(p).stem
        m = re.match(r"^(.*)-(kis|qa|trake)$", stem, re.IGNORECASE)
        if not m:
            print("  !! bỏ qua (hậu tố không hợp lệ):", stem); continue
        qtype = m.group(2).lower()
        raw = open(p, encoding="utf-8").read().strip()
        lines = [l.strip() for l in raw.split("\n") if l.strip()]
        events, ctx = [], []
        for l in lines:
            em = EVENT_RE.match(l)
            if em:
                events.append(em.group(2).strip())
            else:
                ctx.append(l)
        if qtype != "trake":
            events = []
        qs.append(dict(qid=stem, path=p, query_type=qtype, raw=raw,
                       global_context=" ".join(ctx).strip(), events=events,
                       n_events=len(events)))
    return qs

QUERIES = load_queries()
if CFG["LIMIT_QUERIES"]:
    QUERIES = QUERIES[: CFG["LIMIT_QUERIES"]]
print(f"\nloaded {len(QUERIES)} queries:",
      {t: sum(1 for q in QUERIES if q['query_type'] == t) for t in QTYPES})
for q in QUERIES[:3]:
    print(f"  {q['qid']:24s} [{q['query_type']}] events={q['n_events']} :: {q['raw'][:90]}")

# --- rule-based structured parse (fallback / validate cho Qwen) ---
COLOR_VI = {"đỏ": "red", "xanh": "blue/green", "vàng": "yellow", "trắng": "white", "đen": "black",
            "cam": "orange", "tím": "purple", "hồng": "pink", "nâu": "brown", "xám": "gray"}
NUM_VI = {"một": 1, "hai": 2, "ba": 3, "bốn": 4, "năm": 5, "sáu": 6, "bảy": 7, "tám": 8,
          "chín": 9, "mười": 10}
QUESTION_CUE = ("bao nhiêu", "là gì", "màu gì", "ai ", "ở đâu", "khi nào", "tên gì", "?")

def rule_parse(q):
    txt = q["raw"]; low = nfc(txt).lower()
    colors = [en for vi, en in COLOR_VI.items() if vi in low]
    counts = [int(x) for x in re.findall(r"\b(\d{1,3})\b", low)] + \
             [v for k, v in NUM_VI.items() if re.search(rf"\b{k}\b", low)]
    quoted = re.findall(r"[\"“”'‘’]([^\"“”'‘’]{2,40})[\"“”'‘’]", txt)
    upper  = re.findall(r"\b[A-ZĐÀ-Ỹ][A-ZĐÀ-Ỹ0-9]{1,}\b", txt)
    named  = re.findall(r"\b(?:[A-ZĐÀ-Ỹ][a-zà-ỹ]+)(?:\s+[A-ZĐÀ-Ỹ][a-zà-ỹ]+){0,3}", txt)
    question = ""
    if q["query_type"] == "qa":
        sents = re.split(r"(?<=[.?!])\s+", txt.strip())
        cand = [s for s in sents if any(c in nfc(s).lower() for c in QUESTION_CUE)]
        question = (cand[-1] if cand else sents[-1]).strip()
    return dict(
        query_type=q["query_type"],
        q_vi=txt, q_en="", retrieval_queries=[],
        global_context=q["global_context"] or txt,
        visual_cues=[], actions=[], objects=[],
        colors=colors, counts=sorted(set(counts)), people=[], locations=[],
        ocr_terms=list(dict.fromkeys(quoted + [u for u in upper if len(u) > 1]))[:12],
        speech_terms=[], named_entities=list(dict.fromkeys(named))[:12],
        question=question,
        answer_type=("count" if "bao nhiêu" in low else
                     "color" if "màu" in low else
                     "name" if ("tên" in low or " ai " in low) else "short_text"),
        events=q["events"], hard_constraints=[], soft_constraints=[],
        parser="rule",
    )

SCHEMA_KEYS = ["query_type", "q_vi", "q_en", "retrieval_queries", "global_context", "visual_cues", "actions", "objects",
               "colors", "counts", "people", "locations", "ocr_terms", "speech_terms",
               "named_entities", "question", "answer_type", "events",
               "hard_constraints", "soft_constraints"]

def validate_struct(d, q):
    """Ép schema; trả (ok, normalized)."""
    if not isinstance(d, dict):
        return False, None
    # Không chấp nhận parse nửa vời: q_en và bối cảnh phải có nội dung thật.
    if not isinstance(d.get("q_en"), str) or len(d.get("q_en", "").strip()) < 8:
        return False, None
    if not isinstance(d.get("global_context"), str) or len(d.get("global_context", "").strip()) < 8:
        return False, None
    if q["query_type"] == "qa" and not str(d.get("question", "")).strip():
        return False, None
    if q["query_type"] == "trake" and (not isinstance(d.get("events"), list) or len(d.get("events")) != q["n_events"]):
        return False, None
    out = rule_parse(q)
    for k in SCHEMA_KEYS:
        if k in d and d[k] not in (None, ""):
            out[k] = d[k]
    out["query_type"] = q["query_type"]           # nguồn chân lý = hậu tố tên file
    out["q_vi"] = q["raw"]
    for k in ("retrieval_queries", "visual_cues", "actions", "objects", "colors", "people", "locations",
              "ocr_terms", "speech_terms", "named_entities", "hard_constraints", "soft_constraints"):
        v = out.get(k) or []
        out[k] = [str(x) for x in v] if isinstance(v, list) else [str(v)]
    out["counts"] = [int(x) for x in (out.get("counts") or []) if str(x).isdigit()]
    if q["query_type"] == "trake":
        ev = out.get("events") or q["events"]
        out["events"] = [str(x) for x in ev] if ev else q["events"]
        if len(out["events"]) != q["n_events"]:
            out["events"] = q["events"]           # số event phải khớp file đề
    else:
        out["events"] = []
    out["parser"] = d.get("parser", "mimo")
    return True, out

print("\nrule_parse mẫu:")
print(json.dumps({k: v for k, v in rule_parse(QUERIES[0]).items()
                  if k in ("colors", "counts", "ocr_terms", "named_entities", "answer_type")},
                 ensure_ascii=False))

In [ ]:
# ============================================================
# CELL 5 — Model wrappers: SigLIP2 (text+vision), BGE-M3 query, BGE reranker
#   Mỗi wrapper có load()/unload() riêng để kiểm soát VRAM 2×T4.
# ============================================================
from transformers import AutoTokenizer, AutoModel, AutoProcessor, AutoModelForSequenceClassification

def _mdl_path(hf_name, local_sub):
    p = CFG["LOCAL_MODEL_DIR"] and os.path.join(CFG["LOCAL_MODEL_DIR"], local_sub)
    return p if (p and os.path.isdir(p)) else hf_name

def load_hf(cls, name, half=True, **kw):
    """transformers 5.x dùng `dtype=`, 4.x dùng `torch_dtype=` -> thử lần lượt."""
    if half:
        try:
            return cls.from_pretrained(name, dtype=torch.float16, **kw)
        except TypeError:
            return cls.from_pretrained(name, torch_dtype=torch.float16, **kw)
    return cls.from_pretrained(name, **kw)

def as_tensor(out):
    """transformers 5.x trả ModelOutput cho get_text_features/get_image_features,
    4.x trả Tensor -> chuẩn hoá về Tensor embedding."""
    if torch.is_tensor(out):
        return out
    for attr in ("pooler_output", "text_embeds", "image_embeds", "embeds", "last_hidden_state"):
        v = getattr(out, attr, None)
        if v is None and isinstance(out, dict):
            v = out.get(attr)
        if torch.is_tensor(v):
            return v[:, 0] if (attr == "last_hidden_state" and v.dim() == 3) else v
    raise TypeError(f"Không lấy được embedding từ output {type(out)}: "
                    f"{list(getattr(out, 'keys', lambda: [])())}")

# ---------- SigLIP2 ----------
_SIG = {}
def siglip_load(need_vision=False):
    dev = f"cuda:{CFG['GPU_MAIN']}" if torch.cuda.is_available() else "cpu"
    if _SIG.get("model") is not None and (not need_vision or _SIG.get("vision")):
        return
    name = _mdl_path(CFG["SIGLIP"], "siglip2-giant-opt-patch16-384")
    proc = AutoProcessor.from_pretrained(name)
    mdl = load_hf(AutoModel, name).to(dev).eval()
    _SIG.update(proc=proc, model=mdl, dev=dev, vision=True, name=name)
    print("SigLIP2 loaded:", name, "on", dev)

def siglip_unload():
    _SIG.clear(); free_gpu(); print("SigLIP2 unloaded")

@torch.inference_mode()
def siglip_text(texts):
    """Text embedding trong ĐÚNG không gian của keyframe embedding (1536-d, L2-norm)."""
    siglip_load()
    proc, mdl, dev = _SIG["proc"], _SIG["model"], _SIG["dev"]
    enc = proc(text=list(texts), padding="max_length", truncation=True, max_length=64,
               return_tensors="pt").to(dev)
    f = as_tensor(mdl.get_text_features(**enc)).float()
    f = torch.nn.functional.normalize(f, dim=-1)
    v = f.cpu().numpy().astype(np.float32)
    assert v.shape[1] == SIG_INDEX.d, f"SigLIP2 text dim {v.shape[1]} != index {SIG_INDEX.d}"
    assert np.isfinite(v).all()
    return v

@torch.inference_mode()
def siglip_images(pil_images, batch=None):
    siglip_load(need_vision=True)
    proc, mdl, dev = _SIG["proc"], _SIG["model"], _SIG["dev"]
    batch = batch or CFG["REFINE_BATCH"]
    outs, i = [], 0
    while i < len(pil_images):
        b = batch
        while True:
            try:
                enc = proc(images=pil_images[i:i + b], return_tensors="pt").to(dev)
                f = as_tensor(mdl.get_image_features(**enc)).float()
                outs.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy()); break
            except torch.cuda.OutOfMemoryError:
                free_gpu(); b = max(1, b // 2); print("   OOM -> img batch", b)
        i += b
    return np.concatenate(outs).astype(np.float32) if outs else np.zeros((0, SIG_INDEX.d), np.float32)

# ---------- BGE-M3 (query side) ----------
_BGE = {}
def bge_load():
    if _BGE.get("model") is not None: return
    dev = f"cuda:{CFG['GPU_MAIN']}" if torch.cuda.is_available() else "cpu"
    name = _mdl_path(CFG["BGE_M3"], "bge-m3")
    _BGE.update(tok=AutoTokenizer.from_pretrained(name),
                model=load_hf(AutoModel, name).to(dev).eval(), dev=dev)
    print("BGE-M3 loaded")

def bge_unload():
    _BGE.clear(); free_gpu(); print("BGE-M3 unloaded")

@torch.inference_mode()
def bge_query(texts):
    bge_load()
    enc = _BGE["tok"](list(texts), padding=True, truncation=True, max_length=512,
                      return_tensors="pt").to(_BGE["dev"])
    h = as_tensor(_BGE["model"](**enc))
    v = torch.nn.functional.normalize(h.float(), dim=-1).cpu().numpy().astype(np.float32)
    assert v.shape[1] == CAP_INDEX.d
    return v

# ---------- BGE reranker v2-m3 ----------
_RR = {}
def rr_load():
    if _RR.get("model") is not None: return
    dev = f"cuda:{CFG['GPU_AUX'] if torch.cuda.device_count() > 1 else CFG['GPU_MAIN']}" \
          if torch.cuda.is_available() else "cpu"
    name = _mdl_path(CFG["RERANK"], "bge-reranker-v2-m3")
    _RR.update(tok=AutoTokenizer.from_pretrained(name),
               model=load_hf(AutoModelForSequenceClassification, name).to(dev).eval(), dev=dev)
    print("BGE reranker loaded on", dev)

def rr_unload():
    _RR.clear(); free_gpu(); print("reranker unloaded")

@torch.inference_mode()
def rerank(query, docs, batch=8):
    """Chấm query–text-document. KHÔNG dùng để chấm vector ảnh."""
    if not docs: return np.zeros(0, np.float32)
    rr_load()
    out, i = [], 0
    while i < len(docs):
        b = batch
        while True:
            try:
                pairs = [[query, d[:2000] if d else " "] for d in docs[i:i + b]]
                enc = _RR["tok"](pairs, padding=True, truncation=True, max_length=512,
                                 return_tensors="pt").to(_RR["dev"])
                s = _RR["model"](**enc).logits.view(-1).float().cpu().numpy()
                out.append(s); break
            except torch.cuda.OutOfMemoryError:
                free_gpu(); b = max(1, b // 2)
        i += b
    return np.concatenate(out)

# smoke test
_t = siglip_text(["a man in a red shirt speaking at a press conference"])
_D, _I = SIG_INDEX.search(_t, 5)
print("smoke SigLIP2 text->visual:", [(SIG_ROWMAP.video_id[i], int(SIG_ROWMAP.keyframe_n[i]), round(float(s), 3))
                                      for i, s in zip(_I[0], _D[0])])
print("smoke rerank:", np.round(rerank("người mặc áo đỏ",
      ["a man wearing a red shirt", "a recipe for soup"]), 3))
rr_unload()

In [ ]:
# ============================================================
# CELL 6 — MiMo API query planner + Qwen visual/answer tùy chọn (mặc định tắt)
#   MiMo chỉ xử lý text; Qwen không được load khi USE_QWEN_VERIFY/ANSWER=False.
# ============================================================
_QW = {}
def qwen_load():
    if _QW.get("model") is not None: return True
    if _QW.get("failed"):
        return False        # đã thử và thất bại -> KHÔNG thử lại (tránh tải config lặp mỗi query)
    name = _mdl_path(CFG["QWEN"], "qwen3-vl-4b-instruct")
    try:
        from transformers import AutoModelForImageTextToText
        proc = AutoProcessor.from_pretrained(name)
        if CFG["QWEN_4BIT"]:
            from transformers import BitsAndBytesConfig
            mdl = AutoModelForImageTextToText.from_pretrained(
                name, device_map="auto",
                quantization_config=BitsAndBytesConfig(
                    load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                    bnb_4bit_quant_type="nf4")).eval()
        else:
            mdl = load_hf(AutoModelForImageTextToText, name, device_map="auto").eval()
        _QW.update(proc=proc, model=mdl, vision=True, name=name)
        print("Qwen loaded (vision):", name)
    except Exception as e:
        print("!! Qwen vision load lỗi:", repr(e)[:200])
        try:
            from transformers import AutoModelForCausalLM
            tok = AutoTokenizer.from_pretrained(name)
            mdl = load_hf(AutoModelForCausalLM, name, device_map="auto").eval()
            _QW.update(proc=tok, model=mdl, vision=False, name=name)
            print("Qwen loaded (text-only fallback) — visual verification sẽ bị bỏ qua")
        except Exception as e2:
            print("!! Qwen không load được:", repr(e2)[:200],
                  "→ dùng rule parser cho TOÀN BỘ query (sẽ không thử load lại)")
            _QW.clear(); _QW["failed"] = True; return False
    return True

def qwen_unload():
    if not _QW:
        return
    failed = _QW.get("failed")
    _QW.clear()
    if failed:
        _QW["failed"] = True          # giữ cờ qua các stage
    else:
        print("Qwen unloaded")
    free_gpu()

@torch.inference_mode()
def qwen_chat(prompt, images=None, max_new_tokens=512):
    if not qwen_load(): return None
    proc, mdl = _QW["proc"], _QW["model"]
    try:
        if _QW["vision"]:
            content = ([{"type": "image", "image": im} for im in (images or [])] +
                       [{"type": "text", "text": prompt}])
            msgs = [{"role": "user", "content": content}]
            text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            enc = proc(text=[text], images=images or None, return_tensors="pt").to(mdl.device)
        else:
            msgs = [{"role": "user", "content": prompt}]
            text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            enc = proc([text], return_tensors="pt").to(mdl.device)
        out = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[0][enc["input_ids"].shape[1]:]
        dec = proc.batch_decode([gen], skip_special_tokens=True) if hasattr(proc, "batch_decode") else None
        return (dec[0] if dec else proc.decode(gen, skip_special_tokens=True)).strip()
    except torch.cuda.OutOfMemoryError:
        free_gpu(); print("   Qwen OOM -> bỏ qua request này"); return None
    except Exception as e:
        print("   Qwen error:", repr(e)[:160]); return None

PARSE_PROMPT = """Bạn là bộ phân tích truy vấn cho hệ thống truy xuất video tiếng Việt.
Đọc truy vấn và trả về DUY NHẤT một JSON object (không markdown, không giải thích).

Quy tắc dịch q_en: giữ NGUYÊN tên riêng, địa danh, tổ chức, chữ/số cần đọc từ OCR,
số lượng, màu sắc, trang phục, vật thể, quan hệ không gian, THỨ TỰ hành động,
mốc thời gian và phủ định. Không thêm thông tin không có trong truy vấn.

Schema:
{
 "q_en": "bản dịch tiếng Anh có kiểm soát",
 "retrieval_queries": ["3-5 cách diễn đạt ngắn: toàn cảnh, thị giác, OCR/speech, temporal"],
 "global_context": "bối cảnh chung 1-2 câu (tiếng Anh)",
 "visual_cues": ["cue thị giác ngắn, tiếng Anh"],
 "actions": ["hành động theo đúng thứ tự"],
 "objects": ["vật thể"], "colors": ["màu"], "counts": [số nguyên],
 "people": ["mô tả người"], "locations": ["địa điểm"],
 "ocr_terms": ["chữ/số cần khớp OCR, giữ nguyên gốc"],
 "speech_terms": ["từ khoá có thể xuất hiện trong lời nói, tiếng Việt"],
 "named_entities": ["tên riêng"],
 "question": "câu hỏi nếu là Q&A, ngược lại rỗng",
 "answer_type": "count|color|name|yes_no|short_text",
 "events": ["mô tả từng event theo thứ tự, chỉ với TRAKE"],
 "hard_constraints": ["ràng buộc bắt buộc"],
 "soft_constraints": ["ràng buộc mềm (object detection luôn là mềm)"]
}

query_type = %s
%s
TRUY VẤN:
%s

JSON:"""

def extract_json(s):
    if not s: return None
    s = re.sub(r"^```(?:json)?|```$", "", s.strip(), flags=re.MULTILINE).strip()
    i = s.find("{")
    while i >= 0:
        depth = 0
        for j in range(i, len(s)):
            depth += (s[j] == "{") - (s[j] == "}")
            if depth == 0:
                try:
                    return json.loads(s[i:j + 1])
                except Exception:
                    break
        i = s.find("{", i + 1)
    return None

import urllib.request, urllib.error
MIMO_CACHE = OUT / "mimo_cache"
MIMO_CACHE.mkdir(parents=True, exist_ok=True)

MIMO_SYSTEM = (
    "Bạn là query planner cho hệ thống truy xuất video. Chỉ phân tích văn bản; "
    "không xem ảnh, không chọn video/frame và không trả lời câu hỏi. "
    "Trả về duy nhất một JSON object đúng schema. Giữ nguyên tên riêng, OCR, số lượng, "
    "màu sắc, phủ định, quan hệ không gian và thứ tự hành động. Không thêm dữ kiện."
)

def mimo_post(payload):
    if not OPENROUTER_API_KEY:
        raise RuntimeError("Thiếu OPENROUTER_API_KEY")
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    last = None
    for attempt in range(CFG["MIMO_MAX_RETRIES"]):
        req = urllib.request.Request("https://openrouter.ai/api/v1/chat/completions",
            data=body, method="POST", headers={
                "Authorization": "Bearer " + OPENROUTER_API_KEY,
                "Content-Type": "application/json",
                "HTTP-Referer": "https://kaggle.com", "X-Title": "aic2026-nb02"})
        try:
            with urllib.request.urlopen(req, timeout=CFG["MIMO_TIMEOUT"]) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            msg = e.read().decode("utf-8", "replace")[:500]
            last = f"HTTP {e.code}: {msg}"
            if e.code in (400, 401, 403, 404, 422):
                raise RuntimeError(last)
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
        wait = min(20, 2 ** attempt)
        print(f"   MiMo retry {attempt+1}/{CFG['MIMO_MAX_RETRIES']} sau {wait}s: {last[:120]}")
        time.sleep(wait)
    raise RuntimeError("OpenRouter/MiMo lỗi: " + str(last))

def mimo_chat(messages, cache_tag=""):
    payload = {"model": CFG["MIMO_MODEL"], "messages": messages,
               "max_tokens": 1100, "temperature": 0.0,
               "response_format": {"type": "json_object"}}
    key = hashlib.sha1((json.dumps(payload, ensure_ascii=False, sort_keys=True) + cache_tag).encode("utf-8")).hexdigest()
    cp = MIMO_CACHE / key[:2] / f"{key}.json"
    cached = jload(cp, {}) if cp.exists() else {}
    if cached.get("text"):
        return cached["text"]
    if CFG["MIMO_DRY_RUN"]:
        return None
    result = mimo_post(payload)
    text = (((result.get("choices") or [{}])[0].get("message") or {}).get("content") or "").strip()
    if text:
        jdump({"model": CFG["MIMO_MODEL"], "text": text, "usage": result.get("usage") or {}}, cp)
    return text

def parse_query(q, retries=None):
    if not CFG["USE_MIMO_PARSE"] or CFG["MIMO_DRY_RUN"]:
        return rule_parse(q)
    retries = retries or CFG["MIMO_MAX_RETRIES"]
    ev_hint = ""
    if q["query_type"] == "trake":
        ev_hint = ("Truy vấn có đúng %d event, giữ đúng số lượng và thứ tự:\n" % q["n_events"]) + \
                  "\n".join(f"E{i+1}: {e}" for i, e in enumerate(q["events"]))
    user_prompt = PARSE_PROMPT % (q["query_type"], ev_hint, q["raw"])
    messages = [{"role": "system", "content": MIMO_SYSTEM},
                {"role": "user", "content": user_prompt}]
    for attempt in range(retries):
        try:
            raw = mimo_chat(messages, cache_tag=f"|attempt={attempt}")
        except Exception as e:
            print(f"   !! MiMo API lỗi {q['qid']}: {type(e).__name__}: {str(e)[:180]}")
            break
        d = extract_json(raw)
        if d:
            d["parser"] = "mimo"
            ok, norm = validate_struct(d, q)
            if ok:
                if not norm.get("retrieval_queries"):
                    norm["retrieval_queries"] = [norm["q_en"]]
                return norm
        print(f"   MiMo schema chưa hợp lệ {q['qid']} attempt={attempt+1}")
        messages += [{"role": "assistant", "content": (raw or "")[:6000]},
                     {"role": "user", "content": "JSON trên thiếu/sai schema. Hãy trả lại JSON hoàn chỉnh, q_en và global_context không được rỗng."}]
    print(f"   !! parse fallback rule cho {q['qid']}")
    return rule_parse(q)

print("MiMo parser ready | model:", CFG["MIMO_MODEL"], "| dry_run:", CFG["MIMO_DRY_RUN"])

In [ ]:
# ============================================================
# CELL 7 — Stage: parse toàn bộ query (checkpoint từng query)
# ============================================================
PARSED_P = OUT / "review_package" / "queries_parsed.json"
PARSE_META_P = OUT / "review_package" / "queries_parsed_meta.json"
PARSE_SIGNATURE = hashlib.sha1(json.dumps({
    "version": "mimo_query_planner_v1",
    "model": CFG["MIMO_MODEL"],
    "queries": [(q["qid"], q["raw"]) for q in QUERIES],
}, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:16]
PARSED = jload(PARSED_P, {}) or {}
PARSE_META = jload(PARSE_META_P, {}) or {}
if PARSE_META.get("signature") != PARSE_SIGNATURE:
    print("parse checkpoint cũ/không tương thích -> parse lại toàn bộ query")
    PARSED = {}

todo = [q for q in QUERIES if q["qid"] not in PARSED]
print(f"cần parse: {len(todo)}/{len(QUERIES)}")
if todo:
    with Timer("MiMo query parse"):
        for i, q in enumerate(todo):
            check_budget("parse")
            PARSED[q["qid"]] = parse_query(q)
            jdump(PARSED, PARSED_P)
            jdump({"signature": PARSE_SIGNATURE, "n_done": len(PARSED)}, PARSE_META_P)
            print(f"  [{i+1}/{len(todo)}] {q['qid']} parser={PARSED[q['qid']]['parser']}"
                  f" | q_en={PARSED[q['qid']].get('q_en','')[:70]}")
    free_gpu()

parser_counts = pd.Series([v.get("parser", "unknown") for v in PARSED.values()]).value_counts().to_dict()
bad_parse = [q["qid"] for q in QUERIES if q["qid"] not in PARSED or not str(PARSED[q["qid"]].get("q_en", "")).strip()]
print("\nparser distribution:", parser_counts, "| empty q_en:", len(bad_parse))
if CFG["REQUIRE_MIMO_PARSE"] and CFG["USE_MIMO_PARSE"] and bad_parse:
    raise RuntimeError("MiMo parse invalid for: " + ", ".join(bad_parse[:8]) + (" ..." if len(bad_parse) > 8 else ""))
_ex = PARSED[QUERIES[0]["qid"]]
print(json.dumps({k: _ex[k] for k in ("q_en", "visual_cues", "actions", "ocr_terms",
                                      "colors", "counts", "answer_type")},
                 ensure_ascii=False, indent=1))

In [ ]:
# ============================================================
# CELL 8 — Retrieval primitives (mỗi nhánh trả list candidate đã xếp hạng)
# ============================================================
def _keys(video_ids, ns):
    return [f"{v}:{int(n)}" for v, n in zip(video_ids, ns)]

def br_visual(text_list, k=None):
    """SigLIP2 text -> keyframe. Nhiều query text (cue) được max-pool theo candidate."""
    k = k or CFG["K_VISUAL"]
    qv = siglip_text([t for t in text_list if t and t.strip()] or ["scene"])
    D, I = SIG_INDEX.search(qv, k)
    best = {}
    for row_scores, row_ids in zip(D, I):
        for s, r in zip(row_scores, row_ids):
            if r < 0: continue
            key = f"{SIG_ROWMAP.video_id[r]}:{int(SIG_ROWMAP.keyframe_n[r])}"
            best[key] = max(best.get(key, -9.9), float(s))
    return sorted(best.items(), key=lambda x: -x[1])[:k]

def br_dense(index, rowmap, texts, k, level="keyframe"):
    qv = bge_query([t for t in texts if t and t.strip()] or [" "])
    D, I = index.search(qv, k)
    best = {}
    for row_scores, row_ids in zip(D, I):
        for s, r in zip(row_scores, row_ids):
            if r < 0: continue
            if level == "keyframe":
                key = f"{rowmap.video_id[r]}:{int(rowmap.keyframe_n[r])}"
            elif level == "chunk":
                key = f"{rowmap.video_id[r]}#chunk{int(rowmap.chunk_id[r])}"
            else:
                key = str(rowmap.video_id[r])
            best[key] = max(best.get(key, -9.9), float(s))
    return sorted(best.items(), key=lambda x: -x[1])[:k]

def br_bm25(name, terms, k, level="keyframe"):
    if not terms: return []
    q = " ".join(terms)
    bm, meta = BM[name]["bm25"], BM[name]["meta"]
    idx, sc = bm.search({"raw": tokenize(q), "nodia": tokenize_nodia(q)}, topk=k)
    out = []
    for r, s in zip(idx, sc):
        if level == "keyframe":
            key = f"{meta.video_id[r]}:{int(meta.keyframe_n[r])}"
        elif level == "chunk":
            key = f"{meta.video_id[r]}#chunk{int(meta.chunk_id[r])}"
        else:
            key = str(meta.video_id[r])
        out.append((key, float(s)))
    return out

def br_objects(labels, k=400):
    """Object detection = SOFT constraint (có false negative) -> chỉ boost, không loại."""
    if not labels: return []
    inv, meta = OBJ_IDX["inv"], OBJ_IDX["meta"]
    acc = {}
    for lb in labels:
        lb = str(lb).lower().strip()
        arr = inv.get(lb)
        if arr is None:
            hits = [v for kk, v in inv.items() if lb and (lb in kk or kk in lb)]
            arr = np.concatenate(hits) if hits else None
        if arr is None: continue
        for r, s in arr:
            key = f"{meta.video_id[int(r)]}:{int(meta.keyframe_n[int(r)])}"
            acc[key] = acc.get(key, 0.0) + float(s)
    return sorted(acc.items(), key=lambda x: -x[1])[:k]

def chunk_to_frames(chunk_results, per_chunk=3):
    """Map transcript-chunk -> keyframe candidate trong khoảng thời gian chunk."""
    out = {}
    for key, s in chunk_results:
        vid, cid = key.split("#chunk")
        row = TR[(TR.video_id == vid) & (TR.chunk_id == int(cid))]
        if row.empty: continue
        t0, t1 = float(row.start_time.iloc[0]), float(row.end_time.iloc[0])
        kfv = KF[(KF.video_id == vid) & (KF.pts_time >= t0 - 1) & (KF.pts_time <= t1 + 1)]
        if kfv.empty:
            kfv = KF[KF.video_id == vid].iloc[:1]
        mid = (t0 + t1) / 2
        kfv = kfv.assign(_d=(kfv.pts_time - mid).abs()).sort_values("_d").head(per_chunk)
        for _, r in kfv.iterrows():
            kk = f"{vid}:{int(r.keyframe_n)}"
            out[kk] = max(out.get(kk, -9.9), s)
    return sorted(out.items(), key=lambda x: -x[1])

def video_to_prior(video_results):
    if not video_results: return {}
    ss = np.array([s for _, s in video_results], dtype=np.float32)
    lo, hi = float(ss.min()), float(ss.max())
    rng = max(hi - lo, 1e-6)
    return {v: (s - lo) / rng for v, s in video_results}

print("retrieval primitives ready")

In [ ]:
# ============================================================
# CELL 9 — Fusion (weighted RRF) + video prior + temporal NMS + diversity
# ============================================================
def weight_profile(st):
    """Chọn trọng số theo đặc tính query — mỗi profile là baseline có thể tune."""
    n_ocr = len(st.get("ocr_terms") or [])
    n_sp  = len(st.get("speech_terms") or []) + len(st.get("named_entities") or [])
    n_vis = len(st.get("visual_cues") or []) + len(st.get("colors") or []) + len(st.get("objects") or [])
    if n_ocr >= 2 and n_ocr >= n_vis:
        return dict(CFG["W_OCR_HEAVY"]), "ocr_heavy"
    if n_sp >= 3 and n_sp > n_vis:
        return dict(CFG["W_SPEECH"]), "speech_heavy"
    if n_vis >= 3:
        return dict(CFG["W_VISUAL"]), "visual_heavy"
    return dict(CFG["W_DEFAULT"]), "default"

def rrf_fuse(branches, weights, prior=None, rrf_k=None):
    """branches: {name: [(key, score)]} đã xếp hạng. Trả DataFrame candidate."""
    rrf_k = rrf_k or CFG["RRF_K"]
    fused, detail = {}, {}
    for name, res in branches.items():
        w = float(weights.get(name, 0.0))
        if w == 0 or not res: continue
        for rank, (key, s) in enumerate(res):
            add = w / (rrf_k + rank + 1)
            fused[key] = fused.get(key, 0.0) + add
            detail.setdefault(key, {})[name] = round(float(s), 4)
    rows = []
    for key, sc in fused.items():
        vid, n = key.split(":")
        p = float((prior or {}).get(vid, 0.0))
        bonus = min(CFG["VIDEO_PRIOR_W"] * p, CFG["VIDEO_PRIOR_CAP"] * sc)
        rows.append(dict(video_id=vid, keyframe_n=int(n), fused=sc, prior=p,
                         score=sc + bonus,
                         branches=json.dumps(detail.get(key, {}), ensure_ascii=False)))
    df = pd.DataFrame(rows)
    if df.empty: return df
    df["key"] = df.video_id + ":" + df.keyframe_n.astype(str)
    df["frame_idx"] = df.key.map(FI_BY_KEY).fillna(-1).astype(int)
    df["pts_time"] = df.key.map(PTS_BY_KEY)
    return df.sort_values("score", ascending=False).reset_index(drop=True)

def temporal_nms(df, win_sec=None):
    """Gộp candidate cùng video trong cửa sổ thời gian — giữ candidate điểm cao nhất."""
    win = win_sec if win_sec is not None else CFG["NMS_TIME_SEC"]
    keep, taken = [], {}
    for r in df.itertuples(index=False):
        ts = taken.setdefault(r.video_id, [])
        t = r.pts_time if pd.notna(r.pts_time) else -1e9
        if any(abs(t - x) < win for x in ts):
            continue
        ts.append(t); keep.append(r)
    return pd.DataFrame(keep) if keep else df.iloc[:0]

def diversify(df, n=None):
    """Rank1 = confidence cao nhất; top20 giới hạn frame/video; sau đó mở rộng."""
    n = n or CFG["N_SUBMIT"]
    out, per_vid = [], {}
    df = df.reset_index(drop=True)
    for r in df.itertuples(index=False):
        c = per_vid.get(r.video_id, 0)
        cap = CFG["MAX_PER_VIDEO_TOP20"] if len(out) < 20 else CFG["MAX_PER_VIDEO_ALL"]
        if c >= cap:
            continue
        per_vid[r.video_id] = c + 1
        out.append(r)
        if len(out) >= n:
            break
    if len(out) < n:   # bù phần còn lại (giữ đúng tối đa 100 dòng)
        seen = {(r.video_id, r.frame_idx) for r in out}
        for r in df.itertuples(index=False):
            if (r.video_id, r.frame_idx) in seen: continue
            out.append(r); seen.add((r.video_id, r.frame_idx))
            if len(out) >= n: break
    return pd.DataFrame(out).reset_index(drop=True)

def rerank_text(df, q_text, topn=None):
    """BGE reranker chấm query vs caption+OCR của top-N candidate; blend vào score."""
    topn = topn or CFG["RERANK_TOPN"]
    if df.empty: return df
    head = df.head(topn).copy()
    docs = []
    for r in head.itertuples(index=False):
        k = f"{r.video_id}:{r.keyframe_n}"
        docs.append(" | ".join(x for x in [CAP_BY_KEY.get(k, ""), OCR_BY_KEY.get(k, ""),
                                           (SUMMARY_BY_VID.get(r.video_id, "") or "")[:300]] if x))
    sc = rerank(q_text, docs)
    if len(sc) != len(head): return df
    p = 1 / (1 + np.exp(-sc))                       # sigmoid -> [0,1]
    head["rr"] = p
    head["score"] = (1 - CFG["RERANK_W"]) * head["score"] / max(head["score"].max(), 1e-9) \
                    + CFG["RERANK_W"] * p
    tail = df.iloc[topn:].copy()
    if not tail.empty:
        tail["rr"] = np.nan
        tail["score"] = tail["score"] / max(df["score"].max(), 1e-9) * 0.3   # xếp sau block đã rerank
    return pd.concat([head, tail]).sort_values("score", ascending=False).reset_index(drop=True)

print("fusion / nms / diversity / rerank ready")

In [ ]:
# ============================================================
# CELL 10 — Dense frame refinement trên VIDEO GỐC (coarse-to-fine)
#   Keyframe chỉ khoanh vùng; frame_idx cuối cùng phải lấy từ video thật.
# ============================================================
from PIL import Image

def video_meta(vid):
    p = VIDEO_FILE.get(vid)
    if not p: return None
    cap = cv2.VideoCapture(p)
    if not cap.isOpened(): return None
    meta = dict(path=p, fps=float(cap.get(cv2.CAP_PROP_FPS)) or FPS_BY_VID.get(vid, 25.0),
                n_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.release()
    return meta

def read_frames(vid, frame_ids):
    """Đọc frame theo index thật từ video gốc; trả (ids_ok, PIL images)."""
    p = VIDEO_FILE.get(vid)
    if not p: return [], []
    cap = cv2.VideoCapture(p)
    if not cap.isOpened(): return [], []
    ids, imgs = [], []
    for fi in sorted(set(int(x) for x in frame_ids if x is not None and x >= 0)):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ok, fr = cap.read()
        if not ok: continue
        ids.append(fi); imgs.append(Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
    cap.release()
    return ids, imgs

def joint_frame_scores(emb, qv_text):
    """Ưu tiên câu truy vấn đầy đủ, đồng thời yêu cầu các cue cùng hỗ trợ."""
    sim = emb @ qv_text.T
    if sim.shape[1] == 1:
        return sim[:, 0]
    primary = sim[:, 0]                 # q_en đầy đủ; q_vi là nhánh hỗ trợ
    coverage = sim.mean(axis=1)         # tránh frame chỉ khớp đúng một cue
    support = np.sort(sim, axis=1)[:, -min(3, sim.shape[1]):].mean(axis=1)
    return 0.65 * primary + 0.25 * coverage + 0.10 * support

def refine_candidate(vid, center_frame, qv_text, win_sec=None, coarse=None, fine_r=None):
    """Trả (best_frame_idx, best_score, n_decoded). qv_text: (m,1536) SigLIP2 text emb."""
    meta = video_meta(vid)
    if meta is None or center_frame is None or center_frame < 0:
        return int(center_frame or -1), None, 0
    fps = meta["fps"] or 25.0
    win = int((win_sec or CFG["REFINE_WIN_SEC"]) * fps)
    step = coarse or CFG["REFINE_COARSE"]
    fr = fine_r or CFG["REFINE_FINE_R"]
    lo = max(0, center_frame - win); hi = min(meta["n_frames"] - 1, center_frame + win)
    cand = list(range(lo, hi + 1, step))
    ids, imgs = read_frames(vid, cand)
    if not imgs:
        return int(center_frame), None, 0
    emb = siglip_images(imgs)
    sc = joint_frame_scores(emb, qv_text)
    b = int(np.argmax(sc)); best_f, best_s = ids[b], float(sc[b])
    # vòng fine: quét từng frame quanh best
    f_lo, f_hi = max(0, best_f - fr), min(meta["n_frames"] - 1, best_f + fr)
    ids2, imgs2 = read_frames(vid, range(f_lo, f_hi + 1))
    n_dec = len(imgs)
    if imgs2:
        emb2 = siglip_images(imgs2)
        sc2 = joint_frame_scores(emb2, qv_text)
        b2 = int(np.argmax(sc2)); n_dec += len(imgs2)
        if float(sc2[b2]) >= best_s:
            best_f, best_s = ids2[b2], float(sc2[b2])
    return int(best_f), best_s, n_dec

def review_frame_rel(vid, frame_idx):
    """Tên ảnh ổn định, tra cứu được trong review_package."""
    return f"frames/{vid}/{int(frame_idx):06d}.jpg"

def source_keyframe_rel(vid, keyframe_n):
    if keyframe_n is None or pd.isna(keyframe_n):
        return ""
    return f"{vid}/{int(keyframe_n):03d}.jpg"

def contact_sheet(vid, frame_idx, out_path, source_kf_n=None, span=None, cols=5, cell=320):
    """Sheet trước–giữa–sau; đồng thời xuất từng panel thành ảnh có tên chính xác."""
    meta = video_meta(vid)
    if meta is None: return None
    fps = meta["fps"] or 25.0
    span = span if span is not None else int(1.5 * fps)
    targets = np.rint(np.linspace(max(0, frame_idx - span), frame_idx + span, cols)).astype(int).tolist()
    targets[len(targets) // 2] = int(frame_idx)  # luôn có đúng center frame
    targets = list(dict.fromkeys(targets))
    ids, imgs = read_frames(vid, targets)
    if not imgs: return None
    W = cell * len(imgs); sheet = Image.new("RGB", (W, cell + 38), "black")
    from PIL import ImageDraw
    d = ImageDraw.Draw(sheet)
    for i, (fi, im) in enumerate(zip(ids, imgs)):
        rel = review_frame_rel(vid, fi)
        exact_path = OUT / "review_package" / rel
        exact_path.parent.mkdir(parents=True, exist_ok=True)
        if not exact_path.exists():
            im.convert("RGB").save(exact_path, quality=90)
        sheet.paste(im.resize((cell, cell)), (i * cell, 0))
        d.text((i * cell + 4, cell + 2), rel + ("  <= SELECTED" if fi == frame_idx else ""), fill="white")
        src = source_keyframe_rel(vid, source_kf_n)
        if src:
            d.text((i * cell + 4, cell + 19), "source KF: " + src, fill=(180, 220, 255))
    os.makedirs(os.path.dirname(str(out_path)), exist_ok=True)
    sheet.save(out_path, quality=85)
    return dict(sheet=str(out_path), panel_frames=list(map(int, ids)),
                panel_images=[review_frame_rel(vid, x) for x in ids],
                selected_image=review_frame_rel(vid, frame_idx),
                source_keyframe=source_keyframe_rel(vid, source_kf_n))

print("refinement ready | video files:", len(VIDEO_FILE),
      "| WARNING: thiếu video -> refinement bị bỏ qua, dùng frame_idx của keyframe" if not VIDEO_FILE else "")

In [ ]:
# ============================================================
# CELL 11 — KIS / Q&A retrieval: sinh candidate cho một query
# ============================================================
def query_texts(st):
    """Text vào SigLIP2: giữ cả tiếng Việt + tiếng Anh và cue ngắn."""
    base = [st.get("q_en"), st.get("q_vi"), st.get("global_context")]
    variants = (st.get("retrieval_queries") or [])[:5]
    cues = variants + (st.get("visual_cues") or []) + (st.get("actions") or [])
    combo = []
    if st.get("colors") or st.get("objects"):
        combo.append(", ".join((st.get("colors") or []) + (st.get("objects") or [])))
    vals = [str(t).strip() for t in (base + cues[:10] + combo) if t and str(t).strip()]
    return list(dict.fromkeys(vals))[:12]

def dense_texts(st):
    vals = [st.get("q_vi"), st.get("q_en")] + list(st.get("retrieval_queries") or [])[:5] + [
        st.get("global_context"), " ".join(st.get("actions") or [])]
    vals = [str(t).strip() for t in vals if t and str(t).strip()]
    return list(dict.fromkeys(vals))[:8]

def retrieve_kis(st, qid=""):
    vt = query_texts(st)
    branches = {}
    branches["visual"]    = br_visual(vt)
    branches["cap_dense"] = br_dense(CAP_INDEX, CAP_MAP, dense_texts(st), CFG["K_CAP_DENSE"], "keyframe")
    tr_chunks             = br_dense(TR_INDEX, TR_MAP, dense_texts(st) +
                                     [" ".join(st.get("speech_terms") or [])], CFG["K_TR_DENSE"], "chunk")
    branches["tr_dense"]  = chunk_to_frames(tr_chunks)
    branches["bm25_ocr"]  = br_bm25("ocr", (st.get("ocr_terms") or []) +
                                    (st.get("named_entities") or []), CFG["K_BM25_OCR"], "keyframe")
    branches["bm25_cap"]  = br_bm25("caption", (st.get("objects") or []) + (st.get("colors") or []) +
                                    [st.get("q_en") or ""], CFG["K_BM25_CAP"], "keyframe")
    branches["bm25_tr"]   = chunk_to_frames(br_bm25("transcript",
                                    [st.get("q_vi") or ""] + (st.get("speech_terms") or []) +
                                    (st.get("named_entities") or []), CFG["K_BM25_TR"], "chunk"))
    branches["obj"]       = br_objects(st.get("objects") or [])

    # video prior từ summary/metadata (dense + BM25), cộng thêm — không đè frame evidence
    sm_dense = br_dense(SM_INDEX, SM_MAP, dense_texts(st), CFG["K_SM_DENSE"], "video")
    sm_bm    = br_bm25("summary", [st.get("q_en") or "", st.get("q_vi") or ""] +
                       (st.get("named_entities") or []), CFG["K_BM25_SM"], "video")
    prior = video_to_prior(sm_dense)
    for v, s in video_to_prior(sm_bm).items():
        prior[v] = max(prior.get(v, 0.0), 0.8 * s)

    W, profile = weight_profile(st)
    df = rrf_fuse(branches, W, prior)
    if df.empty:
        return df, profile, prior
    df = temporal_nms(df)
    return df, profile, prior

print("KIS/QA retrieval ready")

In [ ]:
# ============================================================
# CELL 12 — TRAKE: video retrieval + temporal alignment (DP đơn điệu + beam)
# ============================================================
def video_rows(vid):
    r = VID_ROWS.get(vid)
    if r is None: return None, None
    a, b, contig = r
    if contig:
        emb = SIG_INDEX.reconstruct_n(a, b - a)
    else:
        rows = SIG_ROWMAP.index[SIG_ROWMAP.video_id == vid].values
        emb = np.stack([SIG_INDEX.reconstruct(int(i)) for i in rows])
        return emb.astype(np.float32), SIG_ROWMAP.loc[rows]
    return np.asarray(emb, dtype=np.float32), SIG_ROWMAP.iloc[a:b]

def trake_candidate_videos(st, topv=None):
    """Chọn video theo độ phủ của từng event, không cộng dồn theo số keyframe."""
    topv = topv or CFG["TRAKE_VIDEOS"]
    contexts = list(dict.fromkeys(str(x).strip() for x in [st.get("q_en"), st.get("q_vi"), st.get("global_context")] if x and str(x).strip()))
    events = list(st.get("events") or [])
    texts = contexts + events
    n_ctx = len(contexts)
    per_text = []
    for text in texts:
        best = {}
        for key, score in br_visual([text], k=CFG["K_VISUAL"]):
            vid = key.split(":")[0]
            best[vid] = max(best.get(vid, -9.9), float(score))
        if best:
            lo, hi = min(best.values()), max(best.values())
            span = max(hi - lo, 1e-6)
            best = {vid: (score - lo) / span for vid, score in best.items()}
        per_text.append(best)
    prior = video_to_prior(br_dense(SM_INDEX, SM_MAP, contexts,
                                      CFG["K_SM_DENSE"], "video"))
    vids = set(prior) | set().union(*(set(x) for x in per_text))
    agg = {}
    for vid in vids:
        ctx_score = max((x.get(vid, 0.0) for x in per_text[:n_ctx]), default=0.0)
        ev = [x.get(vid, 0.0) for x in per_text[n_ctx:]] or [ctx_score]
        coverage = float(np.mean(ev))
        weakest = float(np.min(ev))
        agg[vid] = 0.45 * ctx_score + 0.35 * coverage + 0.10 * weakest + 0.10 * prior.get(vid, 0.0)
    return sorted(agg.items(), key=lambda x: -x[1])[:topv]

def event_scores_in_video(vid, events, st):
    """Trả (kf_meta, S[n_event, n_keyframe]) — điểm SigLIP2 cho từng event tại từng keyframe."""
    emb, meta = video_rows(vid)
    if emb is None or len(emb) == 0: return None, None
    ctx = st.get("global_context") or ""
    qv = siglip_text([f"{ctx}. {e}".strip() if ctx else e for e in events])
    return meta.reset_index(drop=True), (qv @ emb.T)      # (N_event, N_kf)

def dp_align(S, times, min_gap=0.0, beam=None, n_seq=1):
    """Tối ưu Σ event_score với ràng buộc thứ tự đơn điệu t(E1)<...<t(EN).
    Không buộc khoảng cách event bằng nhau; chỉ yêu cầu gap tối thiểu.
    Trả list (total_score, [kf_pos,...]) — beam search giữ nhiều nghiệm."""
    N, M = S.shape
    beam = beam or CFG["TRAKE_BEAM"]
    # khởi tạo với E1
    order = np.argsort(-S[0])[: max(beam * 3, CFG["TRAKE_PER_EVENT"])]
    states = [(float(S[0][i]), [int(i)]) for i in sorted(order)]
    for j in range(1, N):
        nxt = []
        cand = np.argsort(-S[j])[: max(beam * 3, CFG["TRAKE_PER_EVENT"])]
        for sc, path in states:
            last = path[-1]
            for i in sorted(cand):
                if i <= last: continue
                if times is not None and (times[i] - times[last]) < min_gap: continue
                # transition score: ưu tiên tiến về sau nhưng phạt nhảy quá xa
                gap = (times[i] - times[last]) if times is not None else (i - last)
                trans = -0.02 * math.log1p(max(gap, 0.0))
                nxt.append((sc + float(S[j][i]) + trans, path + [int(i)]))
        if not nxt:
            return []
        nxt.sort(key=lambda x: -x[0])
        # giữ đa dạng: không quá 3 nghiệm cùng prefix cuối
        pruned, cnt = [], {}
        for sc, path in nxt:
            k = path[-1]
            if cnt.get(k, 0) >= 3: continue
            cnt[k] = cnt.get(k, 0) + 1
            pruned.append((sc, path))
            if len(pruned) >= beam: break
        states = pruned
    return states[:n_seq]

def retrieve_trake(st, qid=""):
    events = st.get("events") or []
    if not events:
        return pd.DataFrame(), []
    vids = trake_candidate_videos(st)
    seqs = []
    for vid, vscore in vids:
        check_budget("trake")
        meta, S = event_scores_in_video(vid, events, st)
        if S is None: continue
        times = meta.pts_time.values.astype(np.float32)
        sols = dp_align(S, times, min_gap=CFG["TRAKE_MIN_GAP_SEC"],
                        n_seq=CFG["TRAKE_SEQ_PER_VIDEO"])
        for total, path in sols:
            seqs.append(dict(video_id=vid,
                             keyframe_ns=[int(meta.keyframe_n[i]) for i in path],
                             frame_ids=[int(meta.frame_idx[i]) for i in path],
                             pts=[float(meta.pts_time[i]) for i in path],
                             event_scores=[round(float(S[j][path[j]]), 4) for j in range(len(path))],
                             score=float(total) / len(path) + 0.1 * float(vscore)))
    seqs.sort(key=lambda x: -x["score"])
    df = pd.DataFrame([dict(video_id=s["video_id"], score=s["score"],
                            frame_ids=s["frame_ids"], keyframe_ns=s["keyframe_ns"],
                            pts=s["pts"], event_scores=s["event_scores"]) for s in seqs])
    return df, vids

print("TRAKE alignment ready")

In [ ]:
# ============================================================
# CELL 13 — Qwen visual verification + Q&A answering (grounded, <=100 ký tự)
# ============================================================
VERIFY_PROMPT = """Bạn kiểm tra xem khung hình có khớp mô tả truy vấn hay không.
Mô tả: {desc}
OCR trong khung hình: {ocr}
Lời nói quanh thời điểm: {asr}
Trả về JSON duy nhất: {{"match": 0..1, "reason": "ngắn gọn"}}"""

QA_PROMPT = """Bạn trả lời câu hỏi dựa TRÊN ẢNH được cung cấp (khung hình trước–giữa–sau).
Bối cảnh: {ctx}
OCR: {ocr}
Lời nói quanh thời điểm: {asr}
CÂU HỎI: {q}

Yêu cầu:
- Trả lời NGẮN NHẤT có thể, tối đa 100 ký tự, không giải thích.
- Nếu là số lượng thì trả về CHỮ SỐ.
- Giữ nguyên tên riêng / chữ trong ảnh theo đúng nguyên bản (không dịch).
- Nếu ảnh không đủ dữ kiện, đặt confidence thấp.
Trả về JSON duy nhất: {{"answer": "...", "confidence": 0..1, "evidence": "ngắn gọn"}}"""

def asr_near(vid, pts, win=8.0):
    if pd.isna(pts): return ""
    r = TR[(TR.video_id == vid) & (TR.end_time >= pts - win) & (TR.start_time <= pts + win)]
    return " ".join((r.text_vi.fillna("").tolist())[:3])[:600]

def qwen_verify(st, rows, topm=None):
    """Trả dict key -> (match, reason). Chỉ chạy trên top-M để giữ runtime thấp."""
    topm = topm or CFG["QWEN_VERIFY_TOPM"]
    out = {}
    if not CFG["USE_QWEN_VERIFY"] or not qwen_load() or not _QW.get("vision"):
        return out
    desc = st.get("q_en") or st.get("q_vi")
    for r in list(rows)[:topm]:
        check_budget("qwen-verify")
        k = f"{r.video_id}:{r.keyframe_n}"
        ids, imgs = read_frames(r.video_id, [r.frame_idx])
        if not imgs: continue
        txt = qwen_chat(VERIFY_PROMPT.format(desc=desc, ocr=OCR_BY_KEY.get(k, "")[:300],
                                             asr=asr_near(r.video_id, r.pts_time)[:300]),
                        images=imgs, max_new_tokens=160)
        d = extract_json(txt) or {}
        try:
            out[k] = (float(d.get("match", 0.0)), str(d.get("reason", ""))[:200])
        except Exception:
            pass
    return out

def qwen_answer(st, rows, topm=None):
    """Q&A: mỗi candidate một answer riêng (không dùng chung 1 answer cho mọi frame)."""
    topm = topm or CFG["QWEN_QA_TOPM"]
    out = {}
    if not qwen_load():
        return out
    q = st.get("question") or st.get("q_vi")
    for r in list(rows)[:topm]:
        check_budget("qwen-qa")
        k = f"{r.video_id}:{r.keyframe_n}"
        meta = video_meta(r.video_id)
        fps = (meta or {}).get("fps") or FPS_BY_VID.get(r.video_id, 25.0)
        d = int(0.6 * fps)
        ids, imgs = read_frames(r.video_id, [r.frame_idx - d, r.frame_idx, r.frame_idx + d])
        if not imgs and _QW.get("vision"):
            continue
        txt = qwen_chat(QA_PROMPT.format(ctx=(st.get("global_context") or "")[:400],
                                         ocr=OCR_BY_KEY.get(k, "")[:300],
                                         asr=asr_near(r.video_id, r.pts_time)[:400], q=q),
                        images=imgs or None, max_new_tokens=160)
        dd = extract_json(txt) or {}
        ans = str(dd.get("answer", "")).strip().replace("\n", " ")[:100]
        try:
            conf = float(dd.get("confidence", 0.0))
        except Exception:
            conf = 0.0
        if ans:
            out[k] = dict(answer=ans, confidence=conf, evidence=str(dd.get("evidence", ""))[:200])
    return out

print("Qwen verify / QA ready")

In [ ]:
# ============================================================
# CELL 14 — Orchestrator: chạy toàn bộ query, checkpoint từng query
# ============================================================
RUN_SIGNATURE = hashlib.sha1(json.dumps({
    "version": "nb02_mimo_human_review_v1",
    "nb01": MANIFEST01.get("fingerprints"),
    "parse": PARSE_SIGNATURE,
    "models": {k: CFG[k] for k in ("SIGLIP", "BGE_M3", "RERANK", "MIMO_MODEL")},
    "params": {k: CFG[k] for k in CFG if k.startswith(("K_", "W_", "RRF", "NMS", "MAX_", "RERANK_", "MIMO_", "QWEN_", "REFINE_", "TRAKE_", "VIDEO_PRIOR", "N_SUBMIT"))},
}, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:16]
CK = OUT / "ckpt"
def ck_path(qid): return CK / f"q_{qid}.json"

def process_query(q):
    st = PARSED[q["qid"]]
    t0 = time.time()
    rec = dict(qid=q["qid"], query_type=q["query_type"], n_events=q["n_events"],
               _run_signature=RUN_SIGNATURE)

    if q["query_type"] == "trake":
        df, vids = retrieve_trake(st, q["qid"])
        rec["trake_videos"] = [[v, round(float(s), 4)] for v, s in vids]
        if df.empty:
            rec["rows"] = []; rec["runtime_sec"] = round(time.time() - t0, 1); return rec
        df = df.head(CFG["N_SUBMIT"]).reset_index(drop=True)
        df["refined"] = False
        # refinement từng event (frame thật, khoảng GT thường <10 frame)
        if VIDEO_FILE:
            ev_qv = siglip_text([f"{st.get('global_context','')}. {e}".strip()
                                 for e in st["events"]])
            for i in range(min(len(df), max(3, CFG["REFINE_TOPR"] // 2))):
                check_budget("trake-refine")
                newf = []
                for j, fi in enumerate(df.frame_ids[i]):
                    bf, bs, _ = refine_candidate(df.video_id[i], fi, ev_qv[j:j + 1],
                                                 win_sec=1.5, coarse=3, fine_r=5)
                    newf.append(int(bf))
                # Chỉ nhận sequence refine nếu vẫn đúng thứ tự; không tự bịa frame prev+1.
                if all(newf[j] > newf[j - 1] for j in range(1, len(newf))):
                    df.at[i, "frame_ids"] = newf
                    df.at[i, "refined"] = True
                else:
                    print("   TRAKE refine non-monotonic -> giữ sequence gốc", df.video_id[i])
        rec["rows"] = [dict(rank=i + 1, video_id=df.video_id[i], frame_ids=list(map(int, df.frame_ids[i])),
                            keyframe_ns=list(map(int, df.keyframe_ns[i])), pts=df.pts[i],
                            event_scores=df.event_scores[i], score=float(df.score[i]),
                            refined=bool(df.get("refined", pd.Series([False] * len(df)))[i])
                            if "refined" in df else False,
                            source="model")
                       for i in range(len(df))]
        rec["runtime_sec"] = round(time.time() - t0, 1)
        return rec

    # ---- KIS / QA ----
    df, profile, prior = retrieve_kis(st, q["qid"])
    rec["weight_profile"] = profile
    if df.empty:
        rec["rows"] = []; rec["runtime_sec"] = round(time.time() - t0, 1); return rec
    df = rerank_text(df, " | ".join(dense_texts(st)))
    rr_unload()
    df = diversify(df, CFG["N_SUBMIT"])

    # frame refinement trên top-R
    df["refined"] = False
    df["refine_score"] = np.nan
    if VIDEO_FILE:
        qv = siglip_text(query_texts(st))
        for i in range(min(CFG["REFINE_TOPR"], len(df))):
            check_budget("refine")
            bf, bs, nd = refine_candidate(df.video_id[i], int(df.frame_idx[i]), qv)
            if bf >= 0:
                df.at[i, "frame_idx"] = int(bf)
                df.at[i, "refined"] = True
                df.at[i, "refine_score"] = bs
    siglip_unload()

    # Qwen visual verification (KIS + QA) và answer (QA)
    ver = qwen_verify(st, list(df.head(CFG["QWEN_VERIFY_TOPM"]).itertuples(index=False)))
    df["qwen_match"] = [ver.get(f"{r.video_id}:{r.keyframe_n}", (np.nan, ""))[0]
                        for r in df.itertuples(index=False)]
    df["qwen_reason"] = [ver.get(f"{r.video_id}:{r.keyframe_n}", (np.nan, ""))[1]
                         for r in df.itertuples(index=False)]
    # boost theo verification, nhưng KHÔNG loại candidate chưa được verify
    mx = df.score.max() or 1.0
    df["score_final"] = df.score / mx + 0.35 * df.qwen_match.fillna(0.0)
    df = df.sort_values("score_final", ascending=False).reset_index(drop=True)

    df["answer"] = ""
    df["answer_conf"] = np.nan
    df["answer_evidence"] = ""
    if q["query_type"] == "qa" and CFG["USE_QWEN_ANSWER"]:
        ans = qwen_answer(st, list(df.head(CFG["QWEN_QA_TOPM"]).itertuples(index=False)))
        for i, r in enumerate(df.itertuples(index=False)):
            a = ans.get(f"{r.video_id}:{r.keyframe_n}")
            if a:
                df.at[i, "answer"] = a["answer"]
                df.at[i, "answer_conf"] = a["confidence"]
                df.at[i, "answer_evidence"] = a["evidence"]
        # Không sao chép answer sang candidate khác: mỗi answer phải bám đúng frame.
    qwen_unload()

    rec["rows"] = [dict(rank=i + 1, video_id=r.video_id, keyframe_n=int(r.keyframe_n),
                        frame_idx=int(r.frame_idx), pts_time=(None if pd.isna(r.pts_time) else float(r.pts_time)),
                        score=float(r.score_final), fused=float(r.fused), prior=float(r.prior),
                        branches=r.branches, refined=bool(r.refined),
                        qwen_match=(None if pd.isna(r.qwen_match) else float(r.qwen_match)),
                        qwen_reason=r.qwen_reason,
                        answer=r.answer, answer_conf=(None if pd.isna(r.answer_conf) else float(r.answer_conf)),
                        answer_evidence=r.answer_evidence, source="model")
                   for i, r in enumerate(df.itertuples(index=False))]
    rec["runtime_sec"] = round(time.time() - t0, 1)
    return rec

RESULTS = {}
for i, q in enumerate(QUERIES):
    p = ck_path(q["qid"])
    cached = jload(p, {}) if p.exists() else {}
    if cached.get("_run_signature") == RUN_SIGNATURE:
        RESULTS[q["qid"]] = cached; print(f"[{i+1}/{len(QUERIES)}] SKIP {q['qid']} (checkpoint hợp lệ)"); continue
    if p.exists():
        print(f"[{i+1}/{len(QUERIES)}] checkpoint cũ -> chạy lại {q['qid']}")
    print(f"\n=== [{i+1}/{len(QUERIES)}] {q['qid']} [{q['query_type']}] ===", flush=True)
    try:
        rec = process_query(q)
    except TimeoutError as e:
        print("!! BUDGET:", e); break
    except Exception as e:
        traceback.print_exc()
        rec = dict(qid=q["qid"], query_type=q["query_type"], rows=[], error=repr(e)[:400])
    RESULTS[q["qid"]] = rec
    jdump(rec, p)
    print(f"   rows={len(rec.get('rows', []))} runtime={rec.get('runtime_sec')}s "
          f"top1={(rec.get('rows') or [{}])[0].get('video_id')}")

print(f"\nxong {len(RESULTS)}/{len(QUERIES)} query | tổng {(time.time()-T0)/60:.1f} phút")

In [ ]:
# ============================================================
# CELL 15 — Contact sheet + xuất review package (bàn giao NB02 -> NB03)
# ============================================================
P0 = []
for q in QUERIES:
    rec = RESULTS.get(q["qid"], {})
    rows = rec.get("rows") or []
    if rec.get("error"):
        P0.append(f"{q['qid']}: {rec['error']}")
    if not rows:
        P0.append(f"{q['qid']}: không có candidate")
    if len(rows) > CFG["N_SUBMIT"]:
        P0.append(f"{q['qid']}: vượt quá N_SUBMIT")
    if q["query_type"] == "trake":
        for r in rows:
            fs = r.get("frame_ids") or []
            if len(fs) != q["n_events"] or any(fs[j] <= fs[j-1] for j in range(1, len(fs))):
                P0.append(f"{q['qid']} rank {r.get('rank')}: TRAKE frame không hợp lệ")
                break
if P0:
    raise RuntimeError("NB02 validation FAIL:\n- " + "\n- ".join(P0[:20]))
print("NB02 validation PASS | queries=", len(QUERIES))

SHEET_TOP = 12        # đủ rộng cho human review, vẫn giữ thời gian render hợp lý
SHEETS = {}
if VIDEO_FILE:
    siglip_unload()
    with Timer("render contact sheets"):
        for qid, rec in RESULTS.items():
            for r in (rec.get("rows") or [])[:SHEET_TOP]:
                try:
                    if rec["query_type"] == "trake":
                        for j, fi in enumerate(r["frame_ids"]):
                            src_n = int(r["keyframe_ns"][j]) if j < len(r.get("keyframe_ns") or []) else None
                            p = OUT / "review_package" / "sheets" / (
                                f"{qid}_r{r['rank']}_E{j+1}__{r['video_id']}__f{int(fi):06d}"
                                f"__src_{src_n if src_n is not None else 'na'}.jpg")
                            if not p.exists():
                                contact_sheet(r["video_id"], int(fi), p, source_kf_n=src_n)
                            SHEETS.setdefault(qid, {}).setdefault(str(r["rank"]), []).append(str(p.name))
                    else:
                        src_n = r.get("keyframe_n")
                        p = OUT / "review_package" / "sheets" / (
                            f"{qid}_r{r['rank']}__{r['video_id']}__f{int(r['frame_idx']):06d}"
                            f"__src_{int(src_n) if src_n is not None else 'na'}.jpg")
                        if not p.exists():
                            contact_sheet(r["video_id"], int(r["frame_idx"]), p, source_kf_n=src_n)
                        SHEETS.setdefault(qid, {})[str(r["rank"])] = [str(p.name)]
                except Exception as e:
                    print("  sheet err", qid, r.get("rank"), repr(e)[:120])
jdump(SHEETS, OUT / "review_package" / "sheets_index.json")

# --- candidates.parquet (flat, NB03 đọc trực tiếp) ---
flat = []
for qid, rec in RESULTS.items():
    for r in rec.get("rows") or []:
        row = dict(qid=qid, query_type=rec["query_type"], rank=r["rank"],
                   video_id=r["video_id"], score=r.get("score"), source=r.get("source", "model"))
        if rec["query_type"] == "trake":
            row["frame_ids"] = json.dumps(r["frame_ids"])
            row["frame_idx"] = r["frame_ids"][0] if r["frame_ids"] else -1
            row["event_scores"] = json.dumps(r.get("event_scores", []))
            row["image_names"] = json.dumps([review_frame_rel(r["video_id"], x) for x in r["frame_ids"]])
            row["source_keyframe_images"] = json.dumps([source_keyframe_rel(r["video_id"], x) for x in r.get("keyframe_ns", [])])
            row["n_events"] = len(r["frame_ids"])
        else:
            row.update(keyframe_n=r.get("keyframe_n"), frame_idx=r.get("frame_idx"),
                       pts_time=r.get("pts_time"), refined=r.get("refined"),
                       qwen_match=r.get("qwen_match"), qwen_reason=r.get("qwen_reason"),
                       branches=r.get("branches"), answer=r.get("answer", ""),
                       answer_conf=r.get("answer_conf"), answer_evidence=r.get("answer_evidence", ""),
                       image_name=review_frame_rel(r["video_id"], r.get("frame_idx")),
                       source_keyframe_image=source_keyframe_rel(r["video_id"], r.get("keyframe_n")),
                       frame_ids="", event_scores="", image_names="", source_keyframe_images="", n_events=0)
        flat.append(row)
CANDS = pd.DataFrame(flat)
CANDS.to_parquet(OUT / "review_package" / "candidates.parquet", index=False)

# CSV tra cứu ảnh trực tiếp: query/rank/event -> đúng file frame + keyframe nguồn + contact sheet.
frame_catalog = []
for qid, rec in RESULTS.items():
    for r in rec.get("rows") or []:
        sheet_names = (SHEETS.get(qid, {}) or {}).get(str(r["rank"]), [])
        if isinstance(sheet_names, str): sheet_names = [sheet_names]
        if rec["query_type"] == "trake":
            for j, fi in enumerate(r.get("frame_ids") or []):
                kns = r.get("keyframe_ns") or []
                kn = kns[j] if j < len(kns) else None
                frame_catalog.append(dict(qid=qid, query_type=rec["query_type"], rank=r["rank"],
                    event=j + 1, video_id=r["video_id"], frame_idx=int(fi),
                    image_name=review_frame_rel(r["video_id"], fi),
                    source_keyframe_image=source_keyframe_rel(r["video_id"], kn),
                    contact_sheet=(sheet_names[j] if j < len(sheet_names) else "")))
        else:
            frame_catalog.append(dict(qid=qid, query_type=rec["query_type"], rank=r["rank"],
                event="", video_id=r["video_id"], frame_idx=int(r["frame_idx"]),
                image_name=review_frame_rel(r["video_id"], r["frame_idx"]),
                source_keyframe_image=source_keyframe_rel(r["video_id"], r.get("keyframe_n")),
                contact_sheet=(sheet_names[0] if sheet_names else "")))
FRAME_CATALOG = pd.DataFrame(frame_catalog)
FRAME_CATALOG.to_csv(OUT / "review_package" / "frame_catalog.csv", index=False, encoding="utf-8-sig")
print("candidates.parquet rows:", len(CANDS), "| frame_catalog rows:", len(FRAME_CATALOG))
print(CANDS.groupby("query_type")["qid"].nunique().to_dict())

M2 = dict(
    notebook="02_retrieve_refine_candidates_local",
    pipeline_version="nb02_mimo_human_review_v1", run_signature=RUN_SIGNATURE,
    parse_signature=PARSE_SIGNATURE, parser_counts=parser_counts,
    empty_q_en=[qid for qid, st in PARSED.items() if not str(st.get("q_en", "")).strip()],
    created=time.strftime("%Y-%m-%d %H:%M:%S"),
    art_input=str(ART), nb01_fingerprints=MANIFEST01.get("fingerprints"),
    models=dict(siglip=CFG["SIGLIP"], bge_m3=CFG["BGE_M3"], reranker=CFG["RERANK"], mimo=CFG["MIMO_MODEL"],
                parser_api="openrouter", mimo_dry_run=CFG["MIMO_DRY_RUN"]),
    params={k: CFG[k] for k in CFG if k.startswith(("K_", "W_", "RRF", "NMS", "MAX_", "RERANK_",
                                                    "MIMO_", "QWEN_", "REFINE_", "TRAKE_", "N_SUBMIT",
                                                    "VIDEO_PRIOR"))},
    n_queries=len(QUERIES), n_done=len(RESULTS),
    per_query={qid: dict(rows=len(r.get("rows") or []), runtime=r.get("runtime_sec"),
                         error=r.get("error")) for qid, r in RESULTS.items()},
    video_available=bool(VIDEO_FILE),
    runtime_sec=round(time.time() - T0, 1),
)
jdump(M2, OUT / "review_package" / "candidates_manifest.json")
print(f"""
=== BÀN GIAO NB02 -> NB03 ===
{OUT}/review_package/
  candidates.parquet        (tối đa {CFG['N_SUBMIT']} candidate đã xếp hạng / query)
  frame_catalog.csv         (bảng tra query/rank/event -> tên ảnh chính xác)
  queries_parsed.json       (q_vi, q_en, structured constraints)
  sheets/*.jpg + sheets_index.json
  frames/<video_id>/<frame_idx 6 số>.jpg  (từng ảnh chính xác trong contact sheet)
  candidates_manifest.json
Bước tiếp: Save & Version -> NB03 đặt CFG["PKG_INPUT"] = "/kaggle/input/<slug-nb02>/artifacts02_mimo/review_package"
Runtime {(time.time()-T0)/60:.1f} phút | budget còn {budget_left()/3600:.2f} h
""")